<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/notebook_ipynb/shopee_apify_vietnam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Configuration
API_URL = "https://api.apify.com/v2/datasets/Mgs3HI5ccI0gedaFV/items?token=apify_api_YQYYaI77NWc5ympLrHdHEzLUUXn4hH3sBObe"  # @param {type:"string"}
FILENAME = "shopee_vietnam.csv"  # @param {type:"string"}
PRICE_DIVISION_FACTOR = 100_000_000  # @param {type:"number"}

In [ ]:
import requests

try:
    response = requests.get(API_URL)
    response.raise_for_status()
    data = response.json()
    print(f"Successfully fetched {len(data)} items.")
except Exception as e:
    print(f"Error fetching data: {e}")

Successfully fetched 3 items.
First 5 items:
Item 1: {'resultType': 'result', 'batchId': 'batch_1', 'url': 'https://shopee.vn/product/1479423124/26234763697', 'status': 'completed', 'data': {'bff_meta': None, 'error': None, 'error_msg': None, 'data': {'item': {'item_id': 26234763697, 'shop_id': 1479423124, 'item_status': 'normal', 'status': 1, 'item_type': 0, 'reference_item_id': '', 'title': 'Combo Túi Nước giặt Đậm Đặc Hygiene 1.8L x Nước Xả Vải Đậm Đặc Hygiene 2L - Hương Dịu Êm', 'image': 'vn-11134207-81ztc-ms17jmk2nnd0de', 'label_ids': [844931064601283, 844931086908638, 1119699, 1000279, 1883614, 2143672, 2158582, 998171004, 2158587, 998171039, 2153623, 998171042, 2158617, 1400705007, 2153660, 298983321, 298988309, 298988310, 1400705063, 298988329, 700830036, 298983326, 700830086, 298983391, 298973406, 2098694, 2158712, 2153753, 2143693, 2143694, 2143697, 299033331, 2218641, 2213641, 2213649, 2213728, 2243678, 299103349, 299108352, 299108358, 299138315, 2278655, 299138377, 99829601

In [ ]:
import pandas as pd

extracted_data_list = []
for entry in data:
    if 'data' in entry and 'data' in entry['data']:
        extracted_data_list.append(entry['data']['data'])

if extracted_data_list:
    df_raw = pd.DataFrame(extracted_data_list)
    refined_product_data = []

    for index, row in df_raw.iterrows():
        item_data = row.get('item', {})
        product_price_data = row.get('product_price', {})

        if item_data:
            price_dict = product_price_data.get('price')
            price_value = price_dict.get('single_value') if isinstance(price_dict, dict) else None

            pbd_dict = product_price_data.get('price_before_discount')
            pbd_value = pbd_dict.get('single_value') if isinstance(pbd_dict, dict) else None

            refined_product_data.append({
                'item_id': item_data.get('item_id'),
                'shop_id': item_data.get('shop_id'),
                'title': item_data.get('title'),
                'title_tr': item_data.get('title_tr'),
                'currency': item_data.get('currency'),
                'rating_star': item_data.get('item_rating', {}).get('rating_star'),
                'price': price_value,
                'price_before_discount': pbd_value
            })

    df = pd.DataFrame(refined_product_data)

    # Apply division factor from configuration
    df['price'] = df['price'] / PRICE_DIVISION_FACTOR
    df['price_before_discount'] = df['price_before_discount'] / PRICE_DIVISION_FACTOR

    print(f"Successfully created a refined DataFrame with {len(df)} rows.")
    display(df.head())

    # Save to dynamic filename
    df.to_csv(FILENAME, index=False)
    print(f"Saved to {FILENAME}")
else:
    print("No data found.")

Successfully created a refined DataFrame with 3 rows.


,item_id,shop_id,title,title_tr,currency,rating_star,price,price_before_discount
0,26234763697,1479423124,Combo Túi Nước giặt Đậm Đặc Hygiene 1.8L x Nướ...,Combo Bag of Hygiene Concentrated Laundry Dete...,VND,5.000000,308.0,361.0
1,29868227795,1027123033,[Loại 1] Nước rửa chén Thái Lipon F 3200 ch...,[Type 1] Genuine Thai Lipon F 3200 dishwashing...,VND,4.634069,199.0,270.0
2,27770728871,1027123033,[Loại 1] Nước giặt xả đậm đặc Fineline Thái La...,[Type 1] Fineline Thai concentrated laundry de...,VND,4.666667,295.0,500.0


In [ ]:
df.to_csv('shopee_vetnam.csv', index=False)
print("DataFrame successfully saved to 'shopee_vetnam.csv'")

DataFrame successfully saved to 'shopee_vetnam.csv'


In [ ]:
import os
from google.colab import files

# Automatically find all CSV files in the current directory and download them
for file in os.listdir('.'):
    if file.endswith('.csv'):
        print(f'Downloading: {file}')
        files.download(file)

Downloading: shopee_vetnam.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>